# Simple Credit Card Fraud Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dguo97/simple-credit-card-fraud-detection/blob/main/notebooks/fraud_detection_colab.ipynb)

A hybrid fraud detector: an **Isolation Forest** scores how unusual a transaction looks, and that score feeds a **Random Forest** whose probabilities are calibrated. Everything runs on **synthetic data** generated on the fly, so there is nothing to download.

Run the cells in order (Runtime > Run all works too, except the interactive cell in step 3, which waits for you to type).

## 1. Get the code

In [ ]:
!git clone https://github.com/dguo97/simple-credit-card-fraud-detection.git
%cd simple-credit-card-fraud-detection
!pip install -q -r requirements.txt

## 2. Train the model

The first call trains on 20,000 synthetic transactions (a few seconds) and chooses a decision threshold that minimises expected cost, assuming a missed fraud costs 10 times a false alarm.

In [ ]:
import fraud_detection as fd

model = fd.load_or_train()
print(f"Decision threshold: {model.threshold_:.2f}")

## 3. Check your own transaction

Type the values when prompted. The last three questions are optional: press Enter to keep the default. Run the cell again to test another transaction.

In [ ]:
fd.prompt_user_transaction(model)

## 4. Or score transactions in code

Compare an ordinary purchase with a suspicious one (large online purchase at 3am, four transactions in an hour, 400 km from home).

In [ ]:
ordinary = fd.score_transaction(model, 45.50, "Dining", "In-Store", False, hour=19, distance_km=3)
suspicious = fd.score_transaction(
    model, 450, "Electronics", "Online", True, hour=3, txns_last_hour=4, distance_km=400
)
for label, r in (("Ordinary", ordinary), ("Suspicious", suspicious)):
    print(f"{label:<11} fraud probability {r['probability']:.1%}  ->  {'FRAUD' if r['is_fraud'] else 'normal'}")

## 5. Evaluate the model

Trains on 80% of a fresh synthetic dataset, tunes on the training part only, and reports results on the held-out 20%. It takes about a minute. Add `--seeds 5` for mean ± standard deviation over five random datasets (a few minutes).

In [ ]:
!python fraud_detection.py evaluate --plots

In [ ]:
from IPython.display import Image, display

for name in ("confusion_matrix.png", "pr_curve.png", "calibration.png"):
    display(Image(name))